# Buoc 1: Load du lieu

In [4]:
# Import necessary modules
from ssi_fc_data import fc_md_client, model
import pandas as pd  # Import Pandas for DataFrame handling
import json
import config

# Create a Market Data Client
from_date = "08/08/2024"
to_date = "15/11/2024"
client = fc_md_client.MarketDataClient(config)

req = model.daily_ohlc('ACB', from_date, to_date)

data_dict = client.daily_ohlc(config, req)

# Access the list of dictionaries in the "data" field
data_list = data_dict['data']

# Convert the list of dictionaries into a DataFrame
data = pd.DataFrame(data_list, columns=['TradingDate', 'Open', 'High', 'Low', 'Close', 'Volume'])
data = data.rename(columns={'TradingDate': 'Datetime'})

# Print or work with the DataFrame
data

,Datetime,Open,High,Low,Close,Volume
0,08/08/2024,19425,19634,19425,19425,3649600
1,09/08/2024,19551,19718,19467,19718,3006900
2,12/08/2024,19718,19885,19467,19885,4197300
3,13/08/2024,19676,19885,19634,19718,2850000
4,14/08/2024,19801,19885,19676,19676,3316800
...,...,...,...,...,...,...
65,11/11/2024,20804,20888,20595,20762,6742100
66,12/11/2024,20720,20846,20595,20762,4142900
67,13/11/2024,20720,20804,20553,20804,6390100
68,14/11/2024,20595,20762,20512,20512,5795300


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Datetime  70 non-null     object
 1   Open      70 non-null     object
 2   High      70 non-null     object
 3   Low       70 non-null     object
 4   Close     70 non-null     object
 5   Volume    70 non-null     object
dtypes: object(6)
memory usage: 3.4+ KB


# Buoc 2: Viet ham de check chien luoc (Viet 3 ham kiem tra du lieu)

In [4]:
import pandas as pd
# data o tren

data['Datetime'] = pd.to_datetime(data['Datetime'], dayfirst=True)
data.set_index('Datetime', inplace=True)

# Giả sử 'data' là DataFrame của bạn với dữ liệu lịch sử giá cổ phiếu
data['Open'] = pd.to_numeric(data['Open'], errors='coerce')
data['High'] = pd.to_numeric(data['High'], errors='coerce')
data['Low'] = pd.to_numeric(data['Low'], errors='coerce')
data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
data['Volume'] = pd.to_numeric(data['Volume'], errors='coerce')

# Định nghĩa hàm để kiểm tra nến Doji chân dài
def is_long_legged_doji(row):
    body_range = abs(row['Close'] - row['Open']) # Doji khong phan biet Open > Close hay Close > Open
    upper_shadow = row['High'] - max(row['Open'], row['Close'])
    lower_shadow = min(row['Open'], row['Close']) - row['Low']
    # Điều chỉnh ngưỡng này theo dữ liệu cụ thể của bạn
    doji_threshold = 0.1 / 100 * row['Close']
    return body_range <= doji_threshold and upper_shadow >= 2 * body_range and lower_shadow >= 2 * body_range

# Định nghĩa hàm để kiểm tra nến tăng
def is_bullish_candle(current_row, previous_row):
    return (current_row['Close'] > current_row['Open'] and
            current_row['Close'] > previous_row['Close'] and
            previous_row['Close'] <= previous_row['Open'])

# Định nghĩa hàm để kiểm tra nến giảm
def is_bearish_candle(current_row, previous_row):
    return (current_row['Close'] < current_row['Open'] and
            current_row['Close'] < previous_row['Close'] and
            previous_row['Close'] >= previous_row['Open'])

data['Buy_Signal'] = False
data['Sell_Signal'] = False 
    
for i in range(0, len(data)): # Chi lay 2 nen
    current_row = data.iloc[i]
    previous_row = data.iloc[i - 1]
    
    # Kiểm tra nến hiện tại có phải là nến tăng và nếu nến trước đó là nến Doji chân dài
    if is_bullish_candle(current_row, previous_row) and is_long_legged_doji(previous_row):
        # Nếu thỏa mãn cả ba điều kiện, thêm ngày vào danh sách tín hiệu mua
        data.at[current_row.name, 'Buy_Signal'] = True

    if is_bearish_candle(current_row, previous_row) and is_long_legged_doji(previous_row):
        data.at[current_row.name, 'Sell_Signal'] = True

data

,Open,High,Low,Close,Volume,Buy_Signal,Sell_Signal
Datetime,,,,,,,
2024-08-08,23250,23500,23250,23250,3649600,False,False
2024-08-09,23400,23600,23300,23600,3006900,True,False
2024-08-12,23600,23800,23300,23800,4197300,False,False
2024-08-13,23550,23800,23500,23600,2850000,False,False
2024-08-14,23700,23800,23550,23550,3316800,False,False
...,...,...,...,...,...,...,...
2024-11-11,24900,25000,24650,24850,6742100,False,False
2024-11-12,24800,24950,24650,24850,4142900,False,False
2024-11-13,24800,24900,24600,24900,6390100,False,False


# Buoc 3: Show nen len de xem

In [5]:
import plotly.graph_objects as go

# Assuming 'data' is your DataFrame that includes the OHLC data and the Buy/Sell signals
# You will have to replace this with your actual data loading process

# Creating the Candlestick chart for the OHLC data
fig = go.Figure(data=[go.Candlestick(x=data.index,
                                     open=data['Open'],
                                     high=data['High'],
                                     low=data['Low'],
                                     close=data['Close'],
                                     name='OHLC')])

# Adding the Buy signals to the chart
fig.add_trace(go.Scatter(mode='markers',
                         x=data[data['Buy_Signal']].index,
                         y=data[data['Buy_Signal']]['High'],
                         marker=dict(color='green', size=10, symbol='triangle-up'),
                         name='Buy Signal'))

# Adding the Sell signals to the chart
fig.add_trace(go.Scatter(mode='markers',
                         x=data[data['Sell_Signal']].index,
                         y=data[data['Sell_Signal']]['Low'],
                         marker=dict(color='red', size=10, symbol='triangle-down'),
                         name='Sell Signal'))

# Update the layout for a better view
fig.update_layout(title='Stock Price with Buy & Sell Signals',
                  xaxis_title='Date',
                  yaxis_title='Price',
                  xaxis_rangeslider_visible=False)

# Show the figure in your Python environment
fig.show()
